# 🧊 实验 2 —— 随机 FrozenLake 与转移概率

在本实验中，我们将：

1. **可视化“打滑”的冰面环境**，以直观理解随机状态转移的现象。  
2. **将 FrozenLake 建模为一个马尔可夫决策过程（MDP）**，并显式定义转移概率  
   $P(s' \mid s,a)$。  
3. **自行实现并验证转移概率表**，并将结果与 Gym 环境中提供的转移概率进行比较。

### 🧊 演示：打滑与非打滑的 FrozenLake

下面比较 **确定性环境**（`is_slippery=False`）与 **随机环境**（`is_slippery=True`）下的 FrozenLake。

1. 首先创建一个 `is_slippery=False` 的环境，并运行一条轨迹。
2. 然后重新创建一个 `is_slippery=True` 的环境，并再次运行**同一个函数**。
3. 观察在打滑环境中，智能体可能会滑向**非预期方向**，从而产生随机状态转移。

In [ ]:
import gymnasium as gym
import numpy as np
import random
from gymnasium.envs.toy_text.frozen_lake import generate_random_map

In [ ]:
# Deterministic environment (no slipping)
env = gym.make("FrozenLake-v1", map_name="4x4", is_slippery=False, render_mode="ansi")
print("=== Deterministic FrozenLake ===")
obs, info = env.reset(seed=42)
print(env.render())
# Take 4 fixed actions: DOWN, DOWN, RIGHT, RIGHT (1, 1, 2, 2)
for action in [1, 1, 2, 2]:
    obs, reward, terminated, truncated, info = env.step(action)
    print(f"Action: {action}, State: {obs}, Reward: {reward}")
    print(env.render())
    if terminated:
        break

In [ ]:
# 现在重新创建一个具有“打滑”动态的环境
env = gym.make("FrozenLake-v1", map_name="4x4", is_slippery=True, render_mode="ansi")
print("\n=== Slippery FrozenLake ===")
obs, info = env.reset(seed=42)
print(env.render())

# 再次执行相同的固定动作序列
for action in [1, 1, 2, 2]:
    obs, reward, terminated, truncated, info = env.step(action)
    print(f"Action: {action}, State: {obs}, Reward: {reward}")
    print(env.render())
    if terminated:
        break

### 📊 FrozenLake 中“打滑”机制的工作方式

当 `is_slippery=True` 时，环境会引入 **随机状态转移（stochastic transitions）**：

- 每次你选择一个动作时，环境会从以下三种结果中随机采样：
  - **向左偏转**：概率 **1/3**
  - **向前移动（期望方向）**：概率 **1/3**
  - **向右偏转**：概率 **1/3**

这意味着，你所选择的方向实际上只有大约 **33% 的概率**会真正执行成功。

如果某次打滑会使智能体移动到网格之外，那么智能体会 **保持在原位置不动**。

## 🧮 小组练习：将 FrozenLake 建模为四个状态转移矩阵

在打滑模式下，**每个动作都对应一个独立的转移概率矩阵**：

- **`P_LEFT`** —— 执行动作 **LEFT** 时的转移概率
- **`P_DOWN`** —— 执行动作 **DOWN** 时的转移概率
- **`P_RIGHT`** —— 执行动作 **RIGHT** 时的转移概率
- **`P_UP`** —— 执行动作 **UP** 时的转移概率

每个矩阵的大小都是 **9×9**（每一行对应一个当前状态，每一列对应一个下一状态），  
并满足这样的性质：**每一行元素之和等于 1**（如果该状态是终止状态，则该行和可以为 0）。

In [ ]:
# 定义一个更小的 3×3 地图
DESC_3x3 = [
    "SFF",
    "FHF",
    "FFG",
]
env = gym.make("FrozenLake-v1",desc=DESC_3x3,is_slippery=True, render_mode="ansi")
obs, info = env.reset(seed=42)
print(env.render())

In [ ]:
# 9x9 placeholder matrices for you to fill manually

P_LEFT = [
    [0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0],
]

P_DOWN = [
    [0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0],
]

P_RIGHT = [
    [0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0],
]

P_UP = [
    [0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0],
]


In [ ]:
P_all = np.array([P_LEFT, P_DOWN, P_RIGHT, P_UP]).transpose(0, 2, 1)/3
print("组合矩阵的形状:", P_all.shape)  # (4, 9, 9)

### 🎯 概览：从 `P_all` 中采样一条轨迹

现在我们已经得到了 `P_all`，这是一个形状为 `(4, n_states, n_states)` 的组合转移模型。  
这意味着：对于每个动作 `a`，我们都知道从任意当前状态 `s` 转移到每个可能下一状态 `s'` 的概率。

在这一步中，我们将根据以下信息**模拟一条轨迹**：

- 一个起始状态（例如起点 `S` 对应的状态）
- 一个固定的动作序列（例如 `[0, 1, 3, 2]`）

这一次，我们**不再调用** `env.step()`，而是：

- 使用 `P_all[a, s, :]` 读取下一状态的概率分布
- 按照该概率分布随机采样下一个状态
- 重复这一过程，直到所有动作都执行完，并记录访问过的状态序列

这说明，一旦我们拥有了完整的转移模型，  
就可以**完全基于 MDP 表示来生成经验数据**，  
这正是**基于模型的强化学习（model-based reinforcement learning）**中的一个核心思想。

In [ ]:
def sample_trajectory(P_all, start_state, actions):
    s = start_state
    trajectory = [s]
    for a in actions:
        probs = P_all[a, :, s]  # row of probabilities for next states
        s_next = random.choices(range(len(probs)), weights=probs, k=1)[0]
        trajectory.append(s_next)
        s = s_next
    return trajectory

In [ ]:
# Example usage:
obs = 0  # assume starting from state 0 (top-left S)
actions = [0, 1, 3, 2]  # LEFT, DOWN, UP, RIGHT
trajectory = sample_trajectory(P_all, obs, actions)
print("Sampled trajectory of states:", trajectory)

### 🧭 从策略到状态转移矩阵

一旦我们定义了一个**策略（policy）**——即为每个状态指定一个动作——


```python
# Example: choose an action for each of the 9 states
POLICY = [1, 1, 2, 0, 0, 2, 1, 1, 0]  # 0=LEFT, 1=DOWN, 2=RIGHT, 3=UP
```

我们可以利用它构造一个**由该策略决定的状态转移矩阵** `P_pi`。

---

### 📝 我们要做的事情

- 从 `P_all` 开始，它包含了**所有动作**对应的转移概率：
  - 形状为 `(4, 9, 9)` → 4 个动作 × 9 个状态 × 9 个下一状态。
- 对于每一个状态 `s`：
  - 查找策略在该状态选择的动作：`a = POLICY[s]`。
  - 将概率行 `P_all[a, s, :]` 复制到 `P_pi` 中对应的那一行。

In [ ]:
POLICY = [
    0,  # state 0 (top-left)
    0,  # state 1
    0,  # state 2
    0,  # state 3
    0,  # state 4 (center, possibly hole)
    0,  # state 5
    0,  # state 6
    0,  # state 7
    0,  # state 8 (goal state)
]

In [ ]:
n_states = len(POLICY)
P_pi = np.zeros((n_states, n_states))

for s in range(n_states):
    a = POLICY[s]             # action chosen at state s
    P_pi[s, :] = P_all[a, s]  # copy the probabilities for that action

# Print the resulting matrix
import numpy as np
np.set_printoptions(precision=2, suppress=True)
print("Policy-specific transition matrix (P_pi):")
print(P_pi)

### 📈 策略评估：求解状态价值

现在我们已经有：

- **`P_pi`** → 策略对应的状态转移矩阵（9×9）
- **`R`** → 每个状态的奖励向量（或按 state–action–next_state 定义的奖励）

因此，我们可以计算 **在该策略下每个状态的价值（state value）**。

### 📝 我们要做的事情

在给定固定策略的情况下，状态价值函数满足：

$$
V = R + \gamma P_\pi V
$$

其中：

- **`V`** 是状态价值的列向量。
- **`R`** 是每个状态的期望即时奖励列向量。
- **`γ`** 是折扣因子（例如 0.9）。
- **`P_pi`** 是该策略对应的状态转移矩阵。

我们可以将公式整理为：

$$
(I - \gamma P_\pi) V = R
\quad\Rightarrow\quad
V = (I - \gamma P_\pi)^{-1} R
$$

---

### 🎯 你的任务

1. 构建一个长度为 9 的奖励向量 `R`（目标状态为 1，其余为 0）。
2. 选择一个折扣因子 `gamma`（例如 0.9）。
3. 使用 NumPy 求解 `V`。

In [ ]:
# Your time to work on it